# Player Status and News Notebook

This notebook builds a player-availability workflow with **explicit, debuggable API calls** to MLB StatsAPI.

## Important API note

MLB StatsAPI public endpoints used here do **not** require an auth token in normal use.
If calls fail, failures are usually caused by request shape (params/date range/type code), transient service issues, or environment network restrictions.


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import json
import urllib.parse
import urllib.request
import urllib.error

import pandas as pd


## Configuration

Tune these first for easier debugging.


In [ ]:
SEASON = date.today().year
LOOKBACK_DAYS = 14
SPORT_ID = 1

# Safer than relying only on season: explicit date bounds are easier to debug.
START_DATE = date(SEASON, 1, 1)
END_DATE = date.today()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://statsapi.mlb.com/api/v1"


## Request helpers (debug-friendly)

These helpers intentionally return metadata and errors so each call can be inspected.


In [ ]:
def build_url(path: str, params: dict | None = None) -> str:
    params = params or {}
    query = urllib.parse.urlencode(params)
    url = f"{BASE_URL}{path}"
    return f"{url}?{query}" if query else url


def fetch_json_debug(path: str, params: dict | None = None, timeout: int = 60) -> dict:
    url = build_url(path, params)
    result = {
        "ok": False,
        "url": url,
        "status": None,
        "error": None,
        "payload": None,
        "response_text_preview": None,
    }

    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            result["status"] = response.status
            raw_text = response.read().decode("utf-8")
            result["response_text_preview"] = raw_text[:500]
            result["payload"] = json.loads(raw_text)
            result["ok"] = True
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace") if exc.fp else ""
        result["status"] = exc.code
        result["error"] = f"HTTPError: {exc}"
        result["response_text_preview"] = body[:500]
    except urllib.error.URLError as exc:
        result["error"] = f"URLError: {exc}"
    except Exception as exc:
        result["error"] = f"UnexpectedError: {exc}"

    return result


## Step 1: connectivity + endpoint sanity check

This confirms the API host is reachable and returns valid JSON before transaction calls.


In [ ]:
sports_check = fetch_json_debug("/sports", {"sportId": SPORT_ID})

print("OK:", sports_check["ok"])
print("Status:", sports_check["status"])
print("URL:", sports_check["url"])
print("Error:", sports_check["error"])
print("Preview:", sports_check["response_text_preview"])


## Step 2: transaction calls by individual type code

Instead of one combined call, we request each transaction type separately.
That makes failures easy to isolate and debug.


In [ ]:
TRANSACTION_TYPES = [
    "D60",   # 60-day IL
    "D10",   # 10-day IL
    "D15",   # 15-day IL variants
    "DTD",   # day-to-day
    "ACT",   # activated
    "REHAB", # rehab assignment
    "OPT",   # optioned
    "REC",   # recalled
    "DES",   # designated for assignment
    "TR",    # traded
    "SUS",   # suspended
    "RL",    # released
]

common_params = {
    "sportId": SPORT_ID,
    "startDate": START_DATE.isoformat(),
    "endDate": END_DATE.isoformat(),
}

call_results = []
for type_code in TRANSACTION_TYPES:
    params = {**common_params, "transactionTypes": type_code}
    result = fetch_json_debug("/transactions", params)
    result["type_code"] = type_code
    call_results.append(result)

call_log = pd.DataFrame(
    {
        "type_code": [r["type_code"] for r in call_results],
        "ok": [r["ok"] for r in call_results],
        "status": [r["status"] for r in call_results],
        "error": [r["error"] for r in call_results],
        "url": [r["url"] for r in call_results],
    }
)

call_log


In [ ]:
failed_calls = call_log.loc[~call_log["ok"]].copy()
failed_calls


In [ ]:
# Show response previews for any failed type codes.
for item in call_results:
    if not item["ok"]:
        print("\n=== Failed type:", item["type_code"], "===")
        print("Status:", item["status"])
        print("URL:", item["url"])
        print("Error:", item["error"])
        print("Response preview:")
        print(item["response_text_preview"])


## Step 3: combine successful responses


In [ ]:
frames = []
for item in call_results:
    if item["ok"] and item["payload"] is not None:
        records = item["payload"].get("transactions", [])
        if records:
            frame = pd.json_normalize(records)
            frame["requested_type_code"] = item["type_code"]
            frames.append(frame)

transactions = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Loaded {len(transactions):,} rows from {len(frames)} successful type calls.")
transactions.head(10)


In [ ]:
if not transactions.empty:
    rename_map = {
        "person.id": "player_id",
        "person.fullName": "player_name",
        "toTeam.id": "to_team_id",
        "toTeam.name": "to_team_name",
        "fromTeam.id": "from_team_id",
        "fromTeam.name": "from_team_name",
        "typeCode": "type_code",
        "typeDesc": "type_desc",
        "description": "description",
        "date": "transaction_date",
        "effectiveDate": "effective_date",
    }
    transactions = transactions.rename(columns={k: v for k, v in rename_map.items() if k in transactions.columns})

    for dt_col in ["transaction_date", "effective_date"]:
        if dt_col in transactions.columns:
            transactions[dt_col] = pd.to_datetime(transactions[dt_col], errors="coerce")

    transactions = transactions.drop_duplicates(
        subset=[c for c in ["player_id", "transaction_date", "type_code", "description"] if c in transactions.columns]
    ).reset_index(drop=True)

transactions.head(10)


## At-a-glance dashboard (recent changes)


In [ ]:
recent_cutoff = pd.Timestamp(date.today() - timedelta(days=LOOKBACK_DAYS))

dashboard_cols = [
    "transaction_date",
    "player_name",
    "type_desc",
    "description",
    "to_team_name",
    "from_team_name",
]
available_dashboard_cols = [c for c in dashboard_cols if c in transactions.columns]

if not transactions.empty and "transaction_date" in transactions.columns:
    recent_status = (
        transactions.loc[transactions["transaction_date"] >= recent_cutoff, available_dashboard_cols]
        .sort_values("transaction_date", ascending=False)
        .reset_index(drop=True)
    )
else:
    recent_status = pd.DataFrame(columns=available_dashboard_cols)

recent_status.head(50)


## Historical views


In [ ]:
if not transactions.empty and "transaction_date" in transactions.columns:
    daily_volume = (
        transactions.assign(day=transactions["transaction_date"].dt.date)
        .groupby("day", as_index=False)
        .size()
        .rename(columns={"size": "transaction_count"})
    )
else:
    daily_volume = pd.DataFrame(columns=["day", "transaction_count"])

daily_volume.tail(20)


In [ ]:
history_cols = [
    c for c in [
        "transaction_date",
        "player_name",
        "type_desc",
        "description",
        "to_team_name",
        "from_team_name",
        "requested_type_code",
    ] if c in transactions.columns
]

player_status_history = (
    transactions[history_cols]
    .sort_values(["player_name", "transaction_date"], ascending=[True, False])
    .reset_index(drop=True)
    if history_cols else pd.DataFrame()
)

player_status_history.head(100)


## Save outputs


In [ ]:
transactions_out = DATA_DIR / f"player_transactions_{SEASON}.parquet"
recent_out = DATA_DIR / f"player_status_recent_{SEASON}.parquet"
call_log_out = DATA_DIR / f"player_status_call_log_{SEASON}.parquet"

if not transactions.empty:
    transactions.to_parquet(transactions_out, index=False)

if not recent_status.empty:
    recent_status.to_parquet(recent_out, index=False)

call_log.to_parquet(call_log_out, index=False)

print("Wrote:")
print(f" - {transactions_out}")
print(f" - {recent_out}")
print(f" - {call_log_out}")


## Notes

- No token is required for these public MLB StatsAPI endpoints.
- If you get HTTP 400, inspect `call_log` and `response_text_preview`; the notebook now preserves per-type URLs and errors for debugging.
- If your environment blocks outbound requests, you'll see URL/HTTP errors even with valid code.
